In [1]:
"""
Neural Atom Prevalence (NAP) — UCI Concrete Compressive Strength
================================================================
Architecture  : 8 → 50 (BayesianLinear + Swish) → 2 (μ̂, log σ̂)
                One hidden layer, 50 neurons, heteroscedastic regression.
Task          : Predict compressive strength (MPa)
Train/Test    : 9:1 random split, repeated 20 times (seeded)
Reporting     : RMSE and sparsity as  val ± std  over 20 splits
Epochs        : 800 per phase (IMP: 400/round to cap total compute)
LR            : 1e-3  |  Batch : 32  |  S = 1 during training

Pipeline phases (identical 5-phase NAP structure):
  Phase 1 : IMP  — 5 rounds × 400 epochs, 20% prune/round
  Phase 2 : SS-IG on winning ticket (soft gate, 800 epochs)
  Phase 3 : Poisson-Binomial optimal layer-size selection
  Phase 4 : Neural atom selection (ranked π̃)
  Phase 5 : Fine-tune sparse BNN from epoch-0 weights (800 epochs)

Plots (4 panels, mean ± 1σ across 20 runs where applicable):
  A — RMSE vs Epoch  (SS-IG + Fine-tune)
  B — Node Sparsity (π̃ active fraction) vs Epoch
  C — Predictive Uncertainty Decomposition  (median run)
  D — Regression Reliability Diagram         (median run)
"""

import os, math, copy, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats as scipy_stats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ── reproducibility ───────────────────────────────────────────────────────────
BASE_SEED = 42
random.seed(BASE_SEED); np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED); torch.cuda.manual_seed_all(BASE_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING — UCI Concrete Compressive Strength
# ─────────────────────────────────────────────────────────────────────────────

def load_concrete_raw(data_path: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load the Concrete dataset from CSV or Excel.
    Expects the UCI layout: 8 feature columns + 1 target column (last).

    Returns X (N,8)  float32,  y (N,)  float32  (MPa, original scale).
    """
    ext = os.path.splitext(data_path)[1].lower()
    if ext in (".xls", ".xlsx"):
        df = pd.read_excel(data_path)
    else:
        df = pd.read_csv(data_path, sep=None, engine="python")

    # drop any non-numeric header rows that pandas may have kept
    df = df.apply(pd.to_numeric, errors="coerce").dropna()

    y = df.iloc[:, -1].values.astype(np.float32)
    X = df.iloc[:, :-1].values.astype(np.float32)
    print(f"[data] raw shape: X={X.shape}  y={y.shape}  "
          f"target range [{y.min():.1f}, {y.max():.1f}] MPa")
    return X, y


def make_loaders(X: np.ndarray, y: np.ndarray,
                 seed: int, batch_size: int = 32,
                 train_frac: float = 0.9
                 ) -> tuple[DataLoader, DataLoader, float, float]:
    """
    Standardise X and y, split 9:1 with the given seed.
    Returns (train_loader, test_loader, y_mean, y_std).
    y_mean / y_std are used to back-transform RMSE to MPa.
    """
    # per-split standardisation (fit on train, apply to test)
    rng     = np.random.RandomState(seed)
    idx     = rng.permutation(len(y))
    n_train = int(len(y) * train_frac)
    tr_idx, te_idx = idx[:n_train], idx[n_train:]

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_te, y_te = X[te_idx], y[te_idx]

    X_mean, X_std = X_tr.mean(0), X_tr.std(0) + 1e-8
    y_mean, y_std = float(y_tr.mean()), float(y_tr.std())

    X_tr = (X_tr - X_mean) / X_std
    X_te = (X_te - X_mean) / X_std
    y_tr = (y_tr - y_mean) / y_std
    y_te = (y_te - y_mean) / y_std

    def _ds(Xn, yn, shuffle):
        ds = TensorDataset(torch.from_numpy(Xn),
                           torch.from_numpy(yn).unsqueeze(1))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))

    return _ds(X_tr, y_tr, True), _ds(X_te, y_te, False), y_mean, y_std


# ─────────────────────────────────────────────────────────────────────────────
# 2.  NETWORK — 8 → 50 → 2 (one BayesianLinear hidden layer)
# ─────────────────────────────────────────────────────────────────────────────

class SwishActivation(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)


class BayesianLinear(nn.Module):
    """
    Variational BNN layer with spike-and-slab prior.
    Init (paper B.2): μ ~ U(−0.6,0.6), ρ = −6, γ = 0.99.
    """
    def __init__(self, in_features: int, out_features: int, sigma0: float = 1.0):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.sigma0       = sigma0
        self.mu       = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.rho      = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.logit_pi = nn.Parameter(torch.empty(out_features))
        self._init()

    def _init(self):
        nn.init.uniform_(self.mu, -0.6, 0.6)
        nn.init.constant_(self.rho, -6.0)
        nn.init.constant_(self.logit_pi, math.log(0.99 / 0.01))   # logit(0.99)

    def snapshot_epoch0(self):
        self._mu0 = self.mu.data.clone()
        self._rho0 = self.rho.data.clone()
        self._lpi0 = self.logit_pi.data.clone()

    def rewind_to_epoch0(self):
        self.mu.data.copy_(self._mu0)
        self.rho.data.copy_(self._rho0)
        self.logit_pi.data.copy_(self._lpi0)

    @property
    def pi_tilde(self):
        return torch.sigmoid(self.logit_pi)

    def _gate(self, tau, hard):
        if hard:
            return (self.pi_tilde > 0.5).float()
        a = torch.zeros_like(self.logit_pi).uniform_().clamp(1e-6, 1-1e-6)
        return torch.sigmoid((self.logit_pi + a.log() - (1-a).log()) / tau)

    def forward(self, x, tau=0.5, hard=False):
        sig  = F.softplus(self.rho)
        w    = self.mu + sig * torch.randn_like(self.mu)
        out  = F.linear(x, w[:, :-1], w[:, -1])
        return out * self._gate(tau, hard).unsqueeze(0)

    def kl_v(self, prior_pi):
        pi = self.pi_tilde.clamp(1e-6, 1-1e-6)
        p  = torch.tensor(prior_pi, device=pi.device).clamp(1e-6, 1-1e-6)
        return (pi*(pi.log()-p.log()) + (1-pi)*((1-pi).log()-(1-p).log())).sum()

    def kl_weights(self):
        sig  = F.softplus(self.rho)
        s0   = torch.tensor(self.sigma0, device=self.mu.device)
        kl_j = 0.5*((sig/s0).pow(2)+(self.mu/s0).pow(2)-1-2*(sig/s0).log()).sum(1)
        return (self.pi_tilde * kl_j).sum()


class NAPNetworkConcrete(nn.Module):
    """8 → hidden → 2  (heteroscedastic: outputs [μ̂, log σ̂])."""
    def __init__(self, input_dim=8, hidden=50, sigma0=1.0):
        super().__init__()
        self.input_dim = input_dim
        self.hidden    = hidden
        self.layer1    = BayesianLinear(input_dim, hidden, sigma0)
        self.out       = nn.Linear(hidden, 2)
        self.act       = SwishActivation()
        self._reset_out()

    def _reset_out(self):
        nn.init.kaiming_uniform_(self.out.weight, a=math.sqrt(5))
        nn.init.zeros_(self.out.bias)

    def forward(self, x, tau=0.5, hard=False):
        return self.out(self.act(self.layer1(x, tau=tau, hard=hard)))

    def snapshot_epoch0(self):  self.layer1.snapshot_epoch0()
    def rewind_to_epoch0(self):
        self.layer1.rewind_to_epoch0(); self._reset_out()

    def elbo(self, x, y, prior_pi1, n_data, kl_weight=1.0, tau=0.5):
        out     = self.forward(x, tau=tau, hard=False)
        mu_hat  = out[:, 0:1]
        log_sig = out[:, 1:2].clamp(-6.0, 4.0)
        log_lik = (-log_sig - 0.5*(y - mu_hat).pow(2)*torch.exp(-2*log_sig)).sum()
        kl      = self.layer1.kl_v(prior_pi1) + self.layer1.kl_weights()
        return (n_data / x.size(0)) * log_lik - kl_weight * kl

    @torch.no_grad()
    def predict(self, x, n_samples=10):
        """Returns (mean, total_var, aleat_var, epist_var)."""
        means, vars_ = [], []
        for _ in range(n_samples):
            o = self.forward(x, tau=0.5, hard=False)
            means.append(o[:, 0])
            vars_.append(torch.exp(2 * o[:, 1].clamp(-6.0, 4.0)))
        M = torch.stack(means); V = torch.stack(vars_)
        mp, av = M.mean(0), V.mean(0)
        ev = M.var(0)
        return mp, av + ev, av, ev


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PRIOR + SPARSE-NETWORK UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def compute_prior_pi(k_in, k_out, k_next, n, L, B_l=None):
    if B_l is None: B_l = k_out + 1
    pi = min(1.0, math.sqrt(2*math.log(n)*k_next / (n*L*k_in*B_l**2)))
    return float(np.clip(pi, 1e-6, 1-1e-6))


def magnitude_prune(model, prune_frac, mask=None):
    weights, names = [], []
    for name, p in model.named_parameters():
        if "mu" in name and "out" not in name:
            w = p.data.abs()
            if mask and name in mask: w = w * mask[name].float()
            weights.append(w.flatten()); names.append(name)
    flat = torch.cat(weights)
    thr  = torch.quantile(flat[flat > 0], prune_frac)
    new_mask = {}
    for name, p in model.named_parameters():
        if name in names:
            m = p.data.abs() > thr
            if mask and name in mask: m = m & mask[name]
            new_mask[name] = m
        elif mask and name in mask:
            new_mask[name] = mask[name]
    return new_mask


def apply_mask(model, mask):
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in mask: p.data *= mask[name].float()


def surviving_nodes(model, mask):
    mn = "layer1.mu"
    alive = (mask[mn].any(1).sum().item() if mn in mask
             else model.layer1.out_features)
    return {"layer1": int(alive)}


def max_l1_norm(model, mask):
    mu = model.layer1.mu.data
    mn = "layer1.mu"
    if mn in mask: mu = mu * mask[mn].float()
    return {"layer1": float(mu.abs().sum(1).max().item())}


# ─────────────────────────────────────────────────────────────────────────────
# 4.  TRAINING + EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, opt, pi1, n_data,
                kl_w=1.0, tau=0.5, mask=None, hard=False):
    model.train()
    tot = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        loss = -model.elbo(x, y, pi1, n_data, kl_weight=kl_w, tau=tau)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        if mask: apply_mask(model, mask)
        tot += loss.item()
    return tot / len(loader)


@torch.no_grad()
def rmse(model, loader, y_std, n_samples=5):
    model.eval()
    sq = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        mu, *_ = model.predict(x, n_samples)
        sq.append((mu.unsqueeze(1) - y).pow(2).cpu())
    return math.sqrt(torch.cat(sq).mean().item()) * y_std


@torch.no_grad()
def rmse_with_std(model, loader, y_std, n_mc=3):
    """Quick per-epoch RMSE with MC std (n_mc independent gate samples)."""
    model.eval()
    vals = []
    for _ in range(n_mc):
        sq = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x, tau=0.5, hard=False)
            sq.append((out[:, 0:1] - y).pow(2).cpu())
        vals.append(math.sqrt(torch.cat(sq).mean().item()) * y_std)
    return float(np.mean(vals)), float(np.std(vals))


@torch.no_grad()
def full_uq(model, loader, y_std, n_samples=20):
    """Full uncertainty decomposition on test set (original MPa scale)."""
    model.eval()
    yt, yp, ts, als, eps = [], [], [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        mp, tv, av, ev = model.predict(x, n_samples)
        yt.append(y.squeeze(1).cpu().numpy()    * y_std)
        yp.append(mp.cpu().numpy()               * y_std)
        ts.append(tv.sqrt().cpu().numpy()        * y_std)
        als.append(av.sqrt().cpu().numpy()       * y_std)
        eps.append(ev.sqrt().cpu().numpy()       * y_std)
    return (np.concatenate(yt), np.concatenate(yp),
            np.concatenate(ts), np.concatenate(als), np.concatenate(eps))


def _sparsity_snap(model):
    pi = model.layer1.pi_tilde.detach().cpu().numpy()
    return {
        "layer1_frac_active": float((pi > 0.5).mean()),
        "layer1_std"        : float(pi.std()),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 5.  PHASE 1 — IMP
# ─────────────────────────────────────────────────────────────────────────────

def run_imp(model, train_loader, test_loader, n_data, y_std,
            imp_rounds=5, prune_frac=0.20, epochs_per_round=400,
            lr=1e-3, prior_pi1=0.01, verbose=True):
    model.snapshot_epoch0()
    mask = {}
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    surv_per_round = []

    for rnd in range(1, imp_rounds + 1):
        for ep in range(1, epochs_per_round + 1):
            train_epoch(model, train_loader, opt, prior_pi1, n_data,
                        kl_w=1.0, tau=0.5, mask=mask or None, hard=True)
        mask = magnitude_prune(model, prune_frac, mask or None)
        sv   = surviving_nodes(model, mask)
        surv_per_round.append(sv["layer1"])
        if verbose:
            r = rmse(model, test_loader, y_std, n_samples=5)
            print(f"    IMP rnd {rnd}/{imp_rounds} "
                  f"| surviving={sv['layer1']} | RMSE={r:.3f} MPa")
        model.rewind_to_epoch0(); apply_mask(model, mask)
        opt = torch.optim.Adam(model.parameters(), lr=lr)

    return model, mask, surv_per_round


# ─────────────────────────────────────────────────────────────────────────────
# 6.  PHASE 2 — SS-IG (soft gate)
# ─────────────────────────────────────────────────────────────────────────────

def run_ssig(model, train_loader, test_loader, n_data, y_std,
             mask, surv, b_norms, epochs=800, lr=1e-3,
             kl_anneal_ep=320, L=1, verbose=True):
    pi1 = compute_prior_pi(model.input_dim, surv["layer1"], 1,
                           n_data, L, B_l=b_norms["layer1"])
    if verbose:
        print(f"  [SS-IG] prior π₁={pi1:.6f}")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = defaultdict(list)

    for ep in range(1, epochs + 1):
        kl_w = min(1.0, ep / kl_anneal_ep)
        train_epoch(model, train_loader, opt, pi1, n_data,
                    kl_w=kl_w, tau=0.5, mask=mask, hard=False)
        rm, rs = rmse_with_std(model, test_loader, y_std, n_mc=3)
        snap   = _sparsity_snap(model)
        hist["epoch"].append(ep)
        hist["rmse_mean"].append(rm); hist["rmse_std"].append(rs)
        for k, v in snap.items(): hist[k].append(v)
        if verbose and (ep % 100 == 0 or ep == epochs):
            print(f"    ep {ep:4d} | kl_w={kl_w:.2f} | "
                  f"RMSE {rm:.3f}±{rs:.3f} | "
                  f"π̃={model.layer1.pi_tilde.mean().item():.4f}")
    return model, dict(hist)


# ─────────────────────────────────────────────────────────────────────────────
# 7.  PHASE 3-4 — POISSON-BINOMIAL + ATOM SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def poisson_binomial_pmf(probs):
    pmf = np.zeros(len(probs) + 1); pmf[0] = 1.0
    for p in probs.numpy():
        new = pmf.copy()
        new[1:] += pmf[:-1] * p
        new[:-1] *= (1 - p)
        pmf = new
    return pmf


def select_size(model):
    pi  = model.layer1.pi_tilde.detach().cpu()
    pmf = poisson_binomial_pmf(pi)
    k   = max(1, int(pmf.argmax()))
    return {"layer1": k}


def select_atoms(model, k_optimal):
    pi   = model.layer1.pi_tilde.detach().cpu()
    k    = k_optimal["layer1"]
    idxs = torch.argsort(pi, descending=True)[:k].sort().values
    return {"layer1": idxs}


def build_sparse(selected, input_dim=8, sigma0=1.0):
    k1     = len(selected["layer1"])
    sparse = NAPNetworkConcrete(input_dim=input_dim, hidden=k1,
                                sigma0=sigma0).to(DEVICE)
    sparse.snapshot_epoch0()
    return sparse


# ─────────────────────────────────────────────────────────────────────────────
# 8.  PHASE 5 — FINE-TUNE
# ─────────────────────────────────────────────────────────────────────────────

def run_finetune(sparse, train_loader, test_loader, n_data, y_std,
                 k_optimal, b_norms, epochs=800, lr=1e-3,
                 L=1, verbose=True):
    sparse.rewind_to_epoch0()
    pi1 = compute_prior_pi(sparse.input_dim, k_optimal["layer1"], 1,
                           n_data, L, B_l=b_norms["layer1"])
    if verbose:
        print(f"  [FT] prior π₁={pi1:.6f}")
    opt  = torch.optim.Adam(sparse.parameters(), lr=lr)
    hist = defaultdict(list)

    for ep in range(1, epochs + 1):
        train_epoch(sparse, train_loader, opt, pi1, n_data,
                    kl_w=1.0, tau=0.5, mask=None, hard=False)
        rm, rs = rmse_with_std(sparse, test_loader, y_std, n_mc=3)
        snap   = _sparsity_snap(sparse)
        hist["epoch"].append(ep)
        hist["rmse_mean"].append(rm); hist["rmse_std"].append(rs)
        for k, v in snap.items(): hist[k].append(v)
        if verbose and (ep % 100 == 0 or ep == epochs):
            print(f"    ep {ep:4d} | RMSE {rm:.3f}±{rs:.3f}")
    return sparse, dict(hist)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  SINGLE-RUN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_single(X_raw, y_raw, run_seed, y_std_global,
               verbose=False) -> dict:
    """
    Full 5-phase NAP pipeline on one 9:1 split.
    Returns a result dict for aggregation.
    """
    # ── hyperparameters ───────────────────────────────────────────────────────
    INPUT_DIM    = 8
    HIDDEN       = 50
    SIGMA0       = 1.0
    BATCH_SIZE   = 32
    LR           = 1e-3
    IMP_ROUNDS   = 5
    PRUNE_FRAC   = 0.20
    EPOCHS_IMP   = 400        # per round
    EPOCHS_SSIG  = 800
    EPOCHS_FT    = 800
    KL_ANNEAL_EP = 320        # 40% of 800
    PRIOR_PI1_IMP = 0.01

    # ── data split for this run ───────────────────────────────────────────────
    train_loader, test_loader, y_mean, y_std = make_loaders(
        X_raw, y_raw, seed=run_seed, batch_size=BATCH_SIZE)
    n_data = len(train_loader.dataset)

    # ── fresh model ───────────────────────────────────────────────────────────
    torch.manual_seed(run_seed)
    model = NAPNetworkConcrete(INPUT_DIM, HIDDEN, SIGMA0).to(DEVICE)

    # ── Phase 1 ───────────────────────────────────────────────────────────────
    model, mask, surv_per_round = run_imp(
        model, train_loader, test_loader, n_data, y_std,
        imp_rounds=IMP_ROUNDS, prune_frac=PRUNE_FRAC,
        epochs_per_round=EPOCHS_IMP, lr=LR,
        prior_pi1=PRIOR_PI1_IMP, verbose=verbose)

    surv   = surviving_nodes(model, mask)
    b_norm = max_l1_norm(model, mask)

    # ── Phase 2 ───────────────────────────────────────────────────────────────
    model, ssig_hist = run_ssig(
        model, train_loader, test_loader, n_data, y_std,
        mask=mask, surv=surv, b_norms=b_norm,
        epochs=EPOCHS_SSIG, lr=LR,
        kl_anneal_ep=KL_ANNEAL_EP, L=1, verbose=verbose)

    # ── Phase 3 & 4 ───────────────────────────────────────────────────────────
    k_opt    = select_size(model)
    selected = select_atoms(model, k_opt)
    if verbose:
        print(f"  [P3] K^op = {k_opt['layer1']}")

    # ── Phase 5 ───────────────────────────────────────────────────────────────
    sparse = build_sparse(selected, INPUT_DIM, SIGMA0)
    sparse, ft_hist = run_finetune(
        sparse, train_loader, test_loader, n_data, y_std,
        k_optimal=k_opt, b_norms=b_norm,
        epochs=EPOCHS_FT, lr=LR, L=1, verbose=verbose)

    # ── final metrics ─────────────────────────────────────────────────────────
    final_r = rmse(sparse, test_loader, y_std, n_samples=10)
    uq      = full_uq(sparse, test_loader, y_std, n_samples=20)
    sparsity = 1.0 - k_opt["layer1"] / HIDDEN

    return {
        "final_rmse"    : final_r,
        "k_optimal"     : k_opt,
        "sparsity"      : sparsity,
        "surv_per_round": surv_per_round,
        "ssig_history"  : ssig_hist,
        "ft_history"    : ft_hist,
        "uq_data"       : uq,
        "y_std"         : y_std,
        "y_mean"        : y_mean,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 10. MULTI-RUN ORCHESTRATOR — 20 splits
# ─────────────────────────────────────────────────────────────────────────────

def run_20_splits(X_raw, y_raw, n_runs=20, y_std_global=None) -> tuple[list, dict]:
    """
    Run the full NAP pipeline n_runs times on independent 9:1 splits.

    Returns
    -------
    all_results : list of n_runs result dicts
    summary     : dict with mean ± std for key metrics
    """
    print(f"\n{'='*60}")
    print(f"  NAP — Concrete  ·  {n_runs} independent 9:1 splits")
    print(f"  Device : {DEVICE}")
    print(f"{'='*60}")

    all_results = []
    t0 = time.time()

    for run_idx in range(n_runs):
        seed = BASE_SEED + run_idx
        t1   = time.time()
        print(f"\n── Run {run_idx+1:2d}/{n_runs}  (seed={seed}) ──────────────")
        result = run_single(X_raw, y_raw, run_seed=seed,
                            y_std_global=y_std_global, verbose=False)
        elapsed = time.time() - t1
        remaining = elapsed * (n_runs - run_idx - 1)
        print(f"   RMSE={result['final_rmse']:.3f} MPa  "
              f"k*={result['k_optimal']['layer1']}  "
              f"sparsity={result['sparsity']*100:.1f}%  "
              f"[{elapsed:.0f}s, ~{remaining/60:.0f} min left]")
        all_results.append(result)

    # ── aggregate ─────────────────────────────────────────────────────────────
    rmses      = np.array([r["final_rmse"]          for r in all_results])
    k_opts     = np.array([r["k_optimal"]["layer1"] for r in all_results])
    sparsities = np.array([r["sparsity"]             for r in all_results])

    summary = {
        "rmse_mean"      : float(rmses.mean()),
        "rmse_std"       : float(rmses.std()),
        "k_optimal_mean" : float(k_opts.mean()),
        "k_optimal_std"  : float(k_opts.std()),
        "sparsity_mean"  : float(sparsities.mean()),
        "sparsity_std"   : float(sparsities.std()),
        "compression_mean": float(50 / k_opts.mean()),
    }

    total = time.time() - t0
    print(f"\n{'='*60}")
    print(f"  AGGREGATE RESULTS  ({n_runs} runs,  {total/60:.1f} min total)")
    print(f"{'='*60}")
    print(f"  RMSE        :  {summary['rmse_mean']:.3f} ± {summary['rmse_std']:.3f} MPa")
    print(f"  k* neurons  :  {summary['k_optimal_mean']:.1f} ± {summary['k_optimal_std']:.1f}  (of 50)")
    print(f"  Sparsity    :  {summary['sparsity_mean']*100:.1f}% ± {summary['sparsity_std']*100:.1f}%")
    print(f"  Compression :  {summary['compression_mean']:.1f}×  (50 → k* neurons)")
    print(f"{'='*60}")

    return all_results, summary


# ─────────────────────────────────────────────────────────────────────────────
# 11. PLOTTING — 4-panel figure
# ─────────────────────────────────────────────────────────────────────────────

_C = {
    "ssig"  : "#4C72B0",   # steel blue
    "ft"    : "#27AE60",   # emerald
    "aleat" : "#E67E22",   # orange
    "epist" : "#2980B9",   # blue
    "total" : "#7F8C8D",   # grey
    "bg"    : "#F8F9FA",
    "grid"  : "#DEE2E6",
}


def _style(ax, xlabel="", ylabel="", title=""):
    ax.set_facecolor(_C["bg"])
    ax.grid(True, color=_C["grid"], lw=0.7, ls="--", zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    if xlabel: ax.set_xlabel(xlabel, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    if title:  ax.set_title(title, fontsize=11, fontweight="bold", pad=8)


def _band(ax, x, mean, std, color, alpha=0.20):
    ax.fill_between(x, mean - std, mean + std,
                    alpha=alpha, color=color, linewidth=0)


def plot_4_panel(all_results, summary, n_runs=20,
                 save_path="nap_concrete_curves.png"):
    """
    2 × 2 figure aggregating n_runs NAP runs on Concrete.

    Panel A  — RMSE vs Epoch  (SS-IG + FT, mean ± 1σ across runs)
    Panel B  — Node Sparsity vs Epoch  (mean ± 1σ across runs)
    Panel C  — Predictive Uncertainty Decomposition  (median-RMSE run)
    Panel D  — Regression Reliability Diagram         (median-RMSE run)
    """
    # ── identify median-RMSE run ──────────────────────────────────────────────
    rmses       = np.array([r["final_rmse"] for r in all_results])
    median_idx  = int(np.argsort(rmses)[n_runs // 2])
    median_res  = all_results[median_idx]

    # ── stack per-run curves  shape: (n_runs, n_epochs) ──────────────────────
    ssig_rmse = np.array([r["ssig_history"]["rmse_mean"] for r in all_results])
    ft_rmse   = np.array([r["ft_history"]["rmse_mean"]   for r in all_results])
    ssig_spar = np.array([r["ssig_history"]["layer1_frac_active"] for r in all_results])
    ft_spar   = np.array([r["ft_history"]["layer1_frac_active"]   for r in all_results])

    ep_ssig = np.asarray(all_results[0]["ssig_history"]["epoch"])
    ep_ft   = np.asarray(all_results[0]["ft_history"]["epoch"])

    fig, axes = plt.subplots(2, 2, figsize=(15, 12), facecolor="white")
    ax_rmse, ax_spar = axes[0, 0], axes[0, 1]
    ax_uq,   ax_rel  = axes[1, 0], axes[1, 1]

    # ── Panel A — RMSE vs Epoch ───────────────────────────────────────────────
    _style(ax_rmse,
           xlabel="Epoch (within phase)",
           ylabel="Test RMSE (MPa)",
           title=f"A — RMSE vs Epoch  (mean ± 1σ,  {n_runs} splits)")

    for arr, ep, col, lbl, kl_ep in [
        (ssig_rmse, ep_ssig, _C["ssig"], "SS-IG",     320),
        (ft_rmse,   ep_ft,   _C["ft"],   "Fine-tune", None),
    ]:
        mu  = arr.mean(0)
        std = arr.std(0)
        ax_rmse.plot(ep, mu, color=col, lw=2.0, label=lbl, zorder=3)
        _band(ax_rmse, ep, mu, std, col)
        if kl_ep is not None:
            ax_rmse.axvline(kl_ep, color=col, lw=1.0, ls=":",
                            alpha=0.6, zorder=2)
            ax_rmse.text(kl_ep + ep[-1]*0.01,
                         mu.max() * 0.99, "KL=1", fontsize=7,
                         color=col, alpha=0.7)

    ax_rmse.text(0.97, 0.95,
                 f"Final RMSE\n"
                 f"{summary['rmse_mean']:.3f} ± {summary['rmse_std']:.3f} MPa",
                 transform=ax_rmse.transAxes, fontsize=9,
                 ha="right", va="top",
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                           edgecolor="#CCC", alpha=0.9))
    ax_rmse.legend(fontsize=9, loc="upper right")
    ax_rmse.set_xlim(1, max(ep_ssig[-1], ep_ft[-1]))

    # ── Panel B — Node Sparsity vs Epoch ─────────────────────────────────────
    _style(ax_spar,
           xlabel="Epoch (within phase)",
           ylabel="Fraction of Neurons Active  (π̃ > 0.5)",
           title=f"B — Node Sparsity vs Epoch  (mean ± 1σ,  {n_runs} splits)")

    for arr, ep, col, lbl in [
        (ssig_spar, ep_ssig, _C["ssig"], "SS-IG L1"),
        (ft_spar,   ep_ft,   _C["ft"],   "FT L1"),
    ]:
        mu  = arr.mean(0)
        std = arr.std(0)
        ax_spar.plot(ep, mu, color=col, lw=2.0, label=lbl, zorder=3)
        _band(ax_spar, ep, mu, std, col)

    ax_spar.text(0.97, 0.95,
                 f"k* (neurons)\n"
                 f"{summary['k_optimal_mean']:.1f} ± {summary['k_optimal_std']:.1f}  of 50\n"
                 f"Sparsity {summary['sparsity_mean']*100:.1f}% ± "
                 f"{summary['sparsity_std']*100:.1f}%",
                 transform=ax_spar.transAxes, fontsize=9,
                 ha="right", va="top",
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                           edgecolor="#CCC", alpha=0.9))
    ax_spar.set_ylim(-0.05, 1.05)
    ax_spar.legend(fontsize=9, loc="lower left")

    # ── Panel C — Predictive Uncertainty Decomposition ────────────────────────
    y_true, y_pred, tot_s, aleat_s, epist_s = median_res["uq_data"]
    _style(ax_uq,
           xlabel="Uncertainty σ  (MPa)",
           ylabel="Density",
           title=f"C — Predictive Uncertainty Decomposition\n"
                 f"(run {median_idx+1}: RMSE = {rmses[median_idx]:.3f} MPa)")

    clip = np.percentile(tot_s, 99) * 1.05
    bins = np.linspace(0, clip, 55)
    ax_uq.hist(tot_s,   bins=bins, density=True, alpha=0.55,
               color=_C["total"], label="Total σ",    zorder=3)
    ax_uq.hist(aleat_s, bins=bins, density=True, alpha=0.55,
               color=_C["aleat"], label="Aleatoric σ", zorder=3)
    ax_uq.hist(epist_s, bins=bins, density=True, alpha=0.55,
               color=_C["epist"], label="Epistemic σ", zorder=3)
    ax_uq.legend(fontsize=9)

    # epistemic vs aleatoric inset scatter
    ax_in = ax_uq.inset_axes([0.55, 0.38, 0.42, 0.58])
    ax_in.set_facecolor(_C["bg"])
    rng = np.random.default_rng(42)
    idx = rng.choice(len(tot_s), min(1000, len(tot_s)), replace=False)
    ax_in.scatter(aleat_s[idx], epist_s[idx],
                  c=_C["epist"], alpha=0.25, s=5, linewidths=0)
    ax_in.set_xlabel("Aleatoric (MPa)", fontsize=7)
    ax_in.set_ylabel("Epistemic (MPa)", fontsize=7)
    ax_in.set_title("Epist. vs Aleat.", fontsize=7)
    ax_in.tick_params(labelsize=6)
    ax_in.spines[["top", "right"]].set_visible(False)

    ax_uq.text(0.03, 0.97,
               (f"Total σ   = {tot_s.mean():.3f} MPa\n"
                f"Aleat. σ  = {aleat_s.mean():.3f} MPa\n"
                f"Epist. σ  = {epist_s.mean():.3f} MPa\n"
                f"Epist./Total = {epist_s.mean()/tot_s.mean():.3f}"),
               transform=ax_uq.transAxes, fontsize=8, va="top",
               bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                         edgecolor="#CCC", alpha=0.9))

    # ── Panel D — Regression Reliability Diagram ──────────────────────────────
    _style(ax_rel,
           xlabel="Nominal Coverage  (1 − α)",
           ylabel="Observed Coverage",
           title=f"D — Regression Reliability Diagram\n"
                 f"(run {median_idx+1}: median RMSE run)")

    alphas   = np.linspace(0.02, 0.99, 60)
    observed = []
    for a in alphas:
        z   = scipy_stats.norm.ppf(1 - a/2)
        lo  = y_pred - z * tot_s
        hi  = y_pred + z * tot_s
        observed.append(float(np.mean((y_true >= lo) & (y_true <= hi))))
    observed = np.array(observed)
    nominal  = 1 - alphas

    ax_rel.plot([0, 1], [0, 1], "k--", lw=1.3, label="Perfect calibration", zorder=4)
    ax_rel.fill_between(nominal, nominal, observed,
                        where=(observed > nominal),
                        alpha=0.20, color="#2ECC71", label="Over-confident")
    ax_rel.fill_between(nominal, nominal, observed,
                        where=(observed < nominal),
                        alpha=0.20, color="#E74C3C", label="Under-confident")
    ax_rel.plot(nominal, observed, color=_C["ssig"],
                lw=2.2, zorder=3, label="Model coverage")

    cal_err = float(np.mean(np.abs(observed - nominal)))
    ax_rel.text(0.04, 0.95,
                f"Calibration error\n(mean |obs−nom|) = {cal_err:.4f}",
                transform=ax_rel.transAxes, fontsize=9, va="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="#CCC", alpha=0.9))
    ax_rel.set_xlim(0, 1); ax_rel.set_ylim(0, 1)
    ax_rel.legend(fontsize=8, loc="lower right")

    # ── suptitle ──────────────────────────────────────────────────────────────
    fig.suptitle(
        f"Neural Atom Prevalence (NAP) — UCI Concrete Compressive Strength\n"
        f"Architecture: 8 → 50 → 2  ·  {n_runs} × 9:1 splits  ·  "
        f"RMSE = {summary['rmse_mean']:.3f} ± {summary['rmse_std']:.3f} MPa  ·  "
        f"Sparsity = {summary['sparsity_mean']*100:.1f}% ± "
        f"{summary['sparsity_std']*100:.1f}%",
        fontsize=12, fontweight="bold", y=0.998
    )
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n[plot] saved → {save_path}")
    return save_path


# ─────────────────────────────────────────────────────────────────────────────
# 12. SAVE CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────

def save_checkpoint(all_results, summary, n_runs,
                    path="nap_concrete_results.pt"):
    torch.save({
        "summary"    : summary,
        "n_runs"     : n_runs,
        "final_rmses": [r["final_rmse"]          for r in all_results],
        "k_optimals" : [r["k_optimal"]["layer1"] for r in all_results],
        "sparsities" : [r["sparsity"]             for r in all_results],
        "ssig_histories": [r["ssig_history"]       for r in all_results],
        "ft_histories"  : [r["ft_history"]         for r in all_results],
    }, path)
    print(f"[saved] checkpoint → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# 13. ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────
DATA_PATH = "/home/student25/Documents/Essaie/Datasetforthesis/concrete_dataset/Concrete_Data.xls"   
N_RUNS    = 20
# ─────────────────────────────────────────────────────────────────────────────

def run_nap_concrete(data_path: str = DATA_PATH, n_runs: int = N_RUNS):
    """Main entry point — loads data, runs 20 splits, saves plots + checkpoint."""
    X_raw, y_raw = load_concrete_raw(data_path)
    y_std_global = float(y_raw.std())

    all_results, summary = run_20_splits(X_raw, y_raw,
                                          n_runs=n_runs,
                                          y_std_global=y_std_global)

    plot_path = "nap_concrete_curves.png"
    plot_4_panel(all_results, summary, n_runs=n_runs, save_path=plot_path)
    save_checkpoint(all_results, summary, n_runs)

    return all_results, summary


if __name__ == "__main__":
    import argparse as _ap, sys as _sys
    p = _ap.ArgumentParser(description="NAP — Concrete Compressive Strength")
    p.add_argument("--data",   type=str, default=DATA_PATH)
    p.add_argument("--n_runs", type=int, default=N_RUNS)
    args, _ = p.parse_known_args()
    run_nap_concrete(data_path=args.data, n_runs=args.n_runs)

# ── In Jupyter: ───────────────────────────────────────────────────────────────
# all_results, summary = run_nap_concrete("./Concrete_Data.xls", n_runs=20)
# ─────────────────────────────────────────────────────────────────────────────

[data] raw shape: X=(1030, 8)  y=(1030,)  target range [2.3, 82.6] MPa

  NAP — Concrete  ·  20 independent 9:1 splits
  Device : cuda

── Run  1/20  (seed=42) ──────────────
   RMSE=5.361 MPa  k*=13  sparsity=74.0%  [193s, ~61 min left]

── Run  2/20  (seed=43) ──────────────
   RMSE=6.395 MPa  k*=12  sparsity=76.0%  [179s, ~54 min left]

── Run  3/20  (seed=44) ──────────────
   RMSE=6.601 MPa  k*=11  sparsity=78.0%  [186s, ~53 min left]

── Run  4/20  (seed=45) ──────────────
   RMSE=6.616 MPa  k*=13  sparsity=74.0%  [183s, ~49 min left]

── Run  5/20  (seed=46) ──────────────
   RMSE=6.050 MPa  k*=11  sparsity=78.0%  [186s, ~46 min left]

── Run  6/20  (seed=47) ──────────────
   RMSE=6.768 MPa  k*=13  sparsity=74.0%  [187s, ~44 min left]

── Run  7/20  (seed=48) ──────────────
   RMSE=6.046 MPa  k*=11  sparsity=78.0%  [175s, ~38 min left]

── Run  8/20  (seed=49) ──────────────
   RMSE=5.805 MPa  k*=13  sparsity=74.0%  [178s, ~36 min left]

── Run  9/20  (seed=50) ──────────────
 